<a href="https://colab.research.google.com/github/Knguyen351/Data-Mining-Project/blob/main/DataMining_Project_Spring_2026_Alzheimer's_Disease.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Mining Project Template (GitHub + Colab)
## Your Name: Kevin Nguyen
## Date: 4/10/2026
## Project Title: Data-Mining-Project

## Project workflow
This notebook follows an industry-style analytics workflow:

1. **Problem Framing & Data Acquisition**
2. **Exploratory Data Analysis (EDA) & Data Preparation**
3. **Model Development, Evaluation & Business Interpretation**

## GitHub + Colab workflow
1. Create a **new GitHub repository** for your project.
2. Upload this notebook to your repository.
3. In GitHub, open the notebook in **Google Colab**.
4. Commit changes to GitHub as you work.
5. Submit your GitHub repository link when requested.

## Project requirements
- Use a **classification dataset**
- Use **Random Forest** as one of your main models
- Use **Google Colab**
- Include **visualization, preparation, modeling, and interpretation**
- Explain results in a way a manager or stakeholder could understand


In [1]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# AutoViz
!pip install autoviz -q
from autoviz.AutoViz_Class import AutoViz_Class

# scikit-learn tools (Colab-friendly replacement for PyCaret)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report, cohen_kappa_score

# Models to compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Evaluation
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)


# Deliverable 1: Problem Framing & Data Acquisition

## What to include in this markdown cell

- What problem are you trying to solve? Predicttion of Alzheimer's Disease
- What is the **target variable**? Predictor
- What organization, industry, or domain could use this model? Healthcare
- Why does this problem matter? This disease has no cure
- Where did the dataset come from? Kaggle
- Why did you choose this dataset? Enough info to learn and predict



## Data loading options

Choose **one** of the options below:
- load a CSV from GitHub
- upload a CSV into Colab
- read from a direct URL

Keep your original raw data file in your GitHub repository whenever possible.


In [2]:
# Import Dataset
url = "https://raw.githubusercontent.com/Knguyen351/Data-Mining-Project/refs/heads/main/alzheimers_disease_data.csv"
df = pd.read_csv(url)


df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

In [3]:
df.head()

,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,FamilyHistoryAlzheimers,CardiovascularDisease,Diabetes,Depression,HeadInjury,Hypertension,SystolicBP,DiastolicBP,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,MMSE,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,0,0,1,1,0,0,142,72,242.366840,56.150897,33.682563,162.189143,21.463532,6.518877,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,0,0,0,0,0,0,115,64,231.162595,193.407996,79.028477,294.630909,20.613267,7.118696,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,1,0,0,0,0,0,99,116,284.181858,153.322762,69.772292,83.638324,7.356249,5.895077,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,0,0,0,0,0,0,118,115,159.582240,65.366637,68.457491,277.577358,13.991127,8.965106,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,0,0,0,0,0,0,94,117,237.602184,92.869700,56.874305,291.198780,13.517609,6.045039,0,0,0.014691,0,0,1,1,0,0,XXXConfid


# Deliverable 2: Exploratory Data Analysis (EDA) & Data Preparation

## What to include
- basic shape and structure of the data
- variable types
- missing values
- class balance of the target
- visualizations that help explain the data
- preparation steps you used before modeling

## Suggested questions to resolve
- Are there missing values?
- Are the classes balanced?
- Which variables might be useful predictors?
- Are any variables likely to cause problems?
- Do I need to eliminate any variables due to correlation, redundancy, or uniqueness (ex. id)?


The dataset is unbalance
There is a total of 2049 records out those are %65 are no Alzheimer's

In [4]:
df.value_counts("Diagnosis")

,count
Diagnosis,
0,1389
1,760


In [5]:
# Basic data inspection
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)



Shape: (2149, 35)


,PatientID,Age,Gender,Ethnicity,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,SleepQuality,FamilyHistoryAlzheimers,CardiovascularDisease,Diabetes,Depression,HeadInjury,Hypertension,SystolicBP,DiastolicBP,CholesterolTotal,CholesterolLDL,CholesterolHDL,CholesterolTriglycerides,MMSE,FunctionalAssessment,MemoryComplaints,BehavioralProblems,ADL,Confusion,Disorientation,PersonalityChanges,DifficultyCompletingTasks,Forgetfulness,Diagnosis,DoctorInCharge
0,4751,73,0,0,2,22.927749,0,13.297218,6.327112,1.347214,9.025679,0,0,1,1,0,0,142,72,242.366840,56.150897,33.682563,162.189143,21.463532,6.518877,0,0,1.725883,0,0,0,1,0,0,XXXConfid
1,4752,89,0,0,0,26.827681,0,4.542524,7.619885,0.518767,7.151293,0,0,0,0,0,0,115,64,231.162595,193.407996,79.028477,294.630909,20.613267,7.118696,0,0,2.592424,0,0,0,0,1,0,XXXConfid
2,4753,73,0,3,1,17.795882,0,19.555085,7.844988,1.826335,9.673574,1,0,0,0,0,0,99,116,284.181858,153.322762,69.772292,83.638324,7.356249,5.895077,0,0,7.119548,0,1,0,1,0,0,XXXConfid
3,4754,74,1,0,1,33.800817,1,12.209266,8.428001,7.435604,8.392554,0,0,0,0,0,0,118,115,159.582240,65.366637,68.457491,277.577358,13.991127,8.965106,0,1,6.481226,0,0,0,0,0,0,XXXConfid
4,4755,89,0,0,0,20.716974,0,18.454356,6.310461,0.795498,5.597238,0,0,0,0,0,0,94,117,237.602184,92.869700,56.874305,291.198780,13.517609,6.045039,0,0,0.014691,0,0,1,1,0,0,XXXConfid


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2149 entries, 0 to 2148
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   PatientID                  2149 non-null   int64  
 1   Age                        2149 non-null   int64  
 2   Gender                     2149 non-null   int64  
 3   Ethnicity                  2149 non-null   int64  
 4   EducationLevel             2149 non-null   int64  
 5   BMI                        2149 non-null   float64
 6   Smoking                    2149 non-null   int64  
 7   AlcoholConsumption         2149 non-null   float64
 8   PhysicalActivity           2149 non-null   float64
 9   DietQuality                2149 non-null   float64
 10  SleepQuality               2149 non-null   float64
 11  FamilyHistoryAlzheimers    2149 non-null   int64  
 12  CardiovascularDisease      2149 non-null   int64  
 13  Diabetes                   2149 non-null   int64

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
PatientID,2149.0,NaN,NaN,NaN,5825.0,620.507185,4751.0,5288.0,5825.0,6362.0,6899.0
Age,2149.0,NaN,NaN,NaN,74.908795,8.990221,60.0,67.0,75.0,83.0,90.0
Gender,2149.0,NaN,NaN,NaN,0.506282,0.500077,0.0,0.0,1.0,1.0,1.0
Ethnicity,2149.0,NaN,NaN,NaN,0.697534,0.996128,0.0,0.0,0.0,1.0,3.0
EducationLevel,2149.0,NaN,NaN,NaN,1.286645,0.904527,0.0,1.0,1.0,2.0,3.0
BMI,2149.0,NaN,NaN,NaN,27.655697,7.217438,15.008851,21.611408,27.823924,33.869778,39.992767
Smoking,2149.0,NaN,NaN,NaN,0.288506,0.453173,0.0,0.0,0.0,1.0,1.0
AlcoholConsumption,2149.0,NaN,NaN,NaN,10.039442,5.75791,0.002003,5.13981,9.934412,15.157931,19.989293
PhysicalActivity,2149.0,NaN,NaN,NaN,4.920202,2.857191,0.003616,2.570626,4.766424,7.427899,9.987429
DietQuality,2149.0,NaN,NaN,NaN,4.993138,2.909055,0.009385,2.458455,5.076087,7.558625,9.998346


In [6]:
# Missing values summary
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
display(missing_summary)


,0


In [7]:
# TODO: Replace with your actual target column name
target = "Diagnosis"

# Class balance
display(df[target].value_counts(dropna=False))
display(df[target].value_counts(normalize=True, dropna=False))


,count
Diagnosis,
0,1389
1,760


,proportion
Diagnosis,
0,0.646347
1,0.353653


## AutoViz integration

AutoViz is useful for fast exploratory analysis. It can generate many plots at once.

**Important for Colab:** after AutoViz runs, use `plt.close('all')` before creating your own plots later in the notebook. This helps prevent old figures from appearing unexpectedly.


In [8]:
# Install and run AutoViz in Colab if needed
# You may comment this out if AutoViz is already installed in your runtime

# !pip install autoviz

from autoviz.AutoViz_Class import AutoViz_Class
AV = AutoViz_Class()


In [9]:
# AutoViz example
# Replace target with your real target column name before running
# dfte = AV.AutoViz(
#     "",
#     sep=",",
#     depVar=target,
#     dfte=df,
#     header=0,
#     verbose=1,
#     lowess=False,
#     chart_format="svg",
#     max_rows_analyzed=150000,
#     max_cols_analyzed=30
# )

# Clear any queued figures after AutoViz so later plots behave normally in Colab
import matplotlib.pyplot as plt
plt.close('all')


In [10]:
# Code for custom visualiations (optional)

# Example 1: target distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x=target)
plt.title("Target Distribution")
plt.xticks(rotation=45)
plt.show()

# Example 2: numeric histogram for one variable
# Replace 'REPLACE_NUMERIC_COLUMN' with a numeric column from your dataset
# plt.figure(figsize=(6,4))
# sns.histplot(data=df, x='REPLACE_NUMERIC_COLUMN', kde=True)
# plt.title("Distribution of REPLACE_NUMERIC_COLUMN")
# plt.show()

# Example 3: relationship to target
# Replace with columns from your dataset
# plt.figure(figsize=(7,4))
# sns.boxplot(data=df, x=target, y='REPLACE_NUMERIC_COLUMN')
# plt.title("REPLACE_NUMERIC_COLUMN by Target")
# plt.show()


## Data preparation plan

Explain your preparation steps in plain language:
- columns dropped
- missing value handling
- encoding categorical variables
- train/test split strategy
- any feature engineering

Write a short summary in the markdown cell below this one.


### Student preparation summary
Replace this text with your explanation of how you prepared the data.


# Deliverable 3: Model Development, Evaluation & Interpretation

## What to include
- preprocessing pipeline
- Random Forest model
- parameter tuning
- evaluation on the test set
- confusion matrix
- kappa
- feature importance
- interpretation of what the results mean

## Reminder
You should explain results in a business-friendly way, not only with technical language.


One warning though: if your goal is early prediction before diagnosis, then columns like MMSE, MemoryComplaints, FunctionalAssessment, ADL, Confusion, and Forgetfulness may be very strong because they are already symptoms/clinical indicators. That is not wrong, but you need to explain that the model predicts diagnosis using patient health, cognitive, and symptom-related features. (Put in your own words)

“Can we predict Alzheimer’s diagnosis using all available clinical information?”

In [11]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# -----------------------------
# 1. Remove unnecessary columns
# -----------------------------

target = "Diagnosis"

drop_cols = [
    "PatientID",
    "DoctorInCharge",
    "Diagnosis"
]

X = df.drop(columns=drop_cols)
y = df[target]

# -----------------------------
# 2. Check target balance
# -----------------------------

print("Target counts:")
print(y.value_counts())

print("\nTarget percentages:")
print(y.value_counts(normalize=True) * 100)

# -----------------------------
# 3. Train/test split
# -----------------------------
# stratify=y keeps the same 65/35 balance in train and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

print("\nTraining target balance:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target balance:")
print(y_test.value_counts(normalize=True) * 100)

# -----------------------------
# 4. Build balanced Random Forest
# -----------------------------

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=123,
    class_weight="balanced"
)

# Train model
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# -----------------------------
# 5. Evaluate model
# -----------------------------

print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=rf_model.classes_
)

disp.plot()
plt.title("Balanced Random Forest Confusion Matrix")
plt.show()

# -----------------------------
# 6. Feature importance
# -----------------------------

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nFeature Importance:")
print(feature_importance)

# Plot top 15 features
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Top 15 Random Forest Feature Importances")
plt.show()

Target counts:
Diagnosis
0    1389
1     760
Name: count, dtype: int64

Target percentages:
Diagnosis
0    64.634714
1    35.365286
Name: proportion, dtype: float64

Training target balance:
Diagnosis
0    64.630599
1    35.369401
Name: proportion, dtype: float64

Testing target balance:
Diagnosis
0    64.651163
1    35.348837
Name: proportion, dtype: float64

Accuracy: 0.9441860465116279

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       278
           1       0.96      0.88      0.92       152

    accuracy                           0.94       430
   macro avg       0.95      0.93      0.94       430
weighted avg       0.94      0.94      0.94       430


Feature Importance:
            Feature            Importance
23       FunctionalAssessment   0.183154 
26                        ADL   0.159217 
22                       MMSE   0.123723 
24           MemoryComplaints   0.078599 
25         BehavioralProbl

“Can we predict Alzheimer’s risk early before clinical testing or diagnosis?”

In [12]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# -----------------------------
# Target
# -----------------------------

target = "Diagnosis"
y = df[target]

# -----------------------------
# Columns to always remove
# -----------------------------

base_drop_cols = [
    "PatientID",
    "DoctorInCharge",
    "Diagnosis"
]

# -----------------------------
# Possible leakage / clinical assessment columns
# Remove these for early-risk model
# -----------------------------

clinical_assessment_cols = [
    "MMSE",
    "FunctionalAssessment",
    "MemoryComplaints",
    "BehavioralProblems",
    "ADL",
    "Confusion",
    "Disorientation",
    "PersonalityChanges",
    "DifficultyCompletingTasks",
    "Forgetfulness"
]

# -----------------------------
# Function to train and evaluate model
# -----------------------------

def run_random_forest(X, y, model_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=123,
        stratify=y
    )

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=123,
        class_weight="balanced",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1
    )

    rf_model.fit(X_train, y_train)

    y_pred = rf_model.predict(X_test)

    print("=" * 60)
    print(model_name)
    print("=" * 60)

    print("Accuracy:", accuracy_score(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=rf_model.classes_
    )

    disp.plot()
    plt.title(model_name + " Confusion Matrix")
    plt.show()

    feature_importance = pd.DataFrame({
        "Feature": X.columns,
        "Importance": rf_model.feature_importances_
    }).sort_values(by="Importance", ascending=False)

    print("\nTop 15 Feature Importances:")
    print(feature_importance.head(15))

    plt.figure(figsize=(10, 6))
    plt.barh(
        feature_importance.head(15)["Feature"],
        feature_importance.head(15)["Importance"]
    )
    plt.gca().invert_yaxis()
    plt.xlabel("Importance")
    plt.title(model_name + " - Top 15 Feature Importances")
    plt.show()

    return rf_model, feature_importance


# -----------------------------
# Model 1: Full clinical model
# -----------------------------

X_full = df.drop(columns=base_drop_cols)

full_model, full_importance = run_random_forest(
    X_full,
    y,
    "Full Clinical Random Forest Model"
)

# -----------------------------
# Model 2: Early-risk model
# -----------------------------

early_drop_cols = base_drop_cols + clinical_assessment_cols

X_early = df.drop(columns=early_drop_cols)

early_model, early_importance = run_random_forest(
    X_early,
    y,
    "Early-Risk Random Forest Model"
)

Full Clinical Random Forest Model
Accuracy: 0.9441860465116279

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       278
           1       0.96      0.88      0.92       152

    accuracy                           0.94       430
   macro avg       0.95      0.93      0.94       430
weighted avg       0.94      0.94      0.94       430


Top 15 Feature Importances:
            Feature           Importance
23      FunctionalAssessment   0.183154 
26                       ADL   0.159217 
22                      MMSE   0.123723 
24          MemoryComplaints   0.078599 
25        BehavioralProblems   0.053428 
8                DietQuality   0.030762 
21  CholesterolTriglycerides   0.030540 
20            CholesterolHDL   0.030509 
7           PhysicalActivity   0.029972 
9               SleepQuality   0.029820 
19            CholesterolLDL   0.029227 
6         AlcoholConsumption   0.028859 
4                       

## Hyperparameter tuning

We do not know the best settings ahead of time, so we try multiple combinations.

A parameter grid gives the model several choices for each setting. GridSearchCV tests combinations and selects the version that performs best according to the scoring metric.


In [13]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score
)

# -----------------------------
# Target
# -----------------------------

target = "Diagnosis"
y = df[target]

# -----------------------------
# Columns to remove
# -----------------------------

base_drop_cols = [
    "PatientID",
    "DoctorInCharge",
    "Diagnosis"
]

clinical_assessment_cols = [
    "MMSE",
    "FunctionalAssessment",
    "MemoryComplaints",
    "BehavioralProblems",
    "ADL",
    "Confusion",
    "Disorientation",
    "PersonalityChanges",
    "DifficultyCompletingTasks",
    "Forgetfulness"
]

early_drop_cols = base_drop_cols + clinical_assessment_cols

X_early = df.drop(columns=early_drop_cols)

# -----------------------------
# Train/test split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X_early,
    y,
    test_size=0.20,
    random_state=123,
    stratify=y
)

# -----------------------------
# Hyperparameter tuning
# -----------------------------

rf = RandomForestClassifier(
    random_state=123,
    class_weight="balanced_subsample"
)

param_grid = {
    "n_estimators": [300, 500],
    "max_depth": [3, 5, 8, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 3, 5, 10],
    "max_features": ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

# -----------------------------
# Normal predictions
# -----------------------------

y_pred = best_rf.predict(X_test)

print("\nNormal Threshold Results")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=best_rf.classes_
)

disp.plot()
plt.title("Early-Risk Random Forest - Normal Threshold")
plt.show()

# -----------------------------
# Predicted probabilities
# -----------------------------

y_prob = best_rf.predict_proba(X_test)[:, 1]

print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

# -----------------------------
# Adjust threshold
# -----------------------------
# Lower threshold = catches more Alzheimer’s cases
# Higher threshold = fewer false alarms

threshold = 0.35

y_pred_adjusted = (y_prob >= threshold).astype(int)

print("\nAdjusted Threshold Results")
print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, y_pred_adjusted))
print(classification_report(y_test, y_pred_adjusted))

cm_adjusted = confusion_matrix(y_test, y_pred_adjusted)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_adjusted,
    display_labels=best_rf.classes_
)

disp.plot()
plt.title("Early-Risk Random Forest - Adjusted Threshold")
plt.show()

# -----------------------------
# Feature importance
# -----------------------------

feature_importance = pd.DataFrame({
    "Feature": X_early.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("\nTop 15 Feature Importances:")
print(feature_importance.head(15))

plt.figure(figsize=(10, 6))
plt.barh(
    feature_importance.head(15)["Feature"],
    feature_importance.head(15)["Importance"]
)
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title("Early-Risk Model - Top 15 Feature Importances")
plt.show()

Fitting 5 folds for each of 240 candidates, totalling 1200 fits
Best Parameters:
{'max_depth': 3, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'min_samples_split': 2, 'n_estimators': 300}

Normal Threshold Results
Accuracy: 0.541860465116279
              precision    recall  f1-score   support

           0       0.67      0.57      0.62       278
           1       0.38      0.49      0.43       152

    accuracy                           0.54       430
   macro avg       0.53      0.53      0.52       430
weighted avg       0.57      0.54      0.55       430

ROC-AUC Score: 0.5144831503218478

Adjusted Threshold Results
Threshold: 0.35
Accuracy: 0.35348837209302325
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       278
           1       0.35      1.00      0.52       152

    accuracy                           0.35       430
   macro avg       0.18      0.50      0.26       430
weighted avg       0.12      0.35      0.18     

The early-risk model had about 65% accuracy, but the classification report showed that it mainly predicted the majority class. The recall for Alzheimer’s cases was very low, meaning the model did not identify most positive diagnosis cases. This suggests that lifestyle and general health variables alone are not strong enough to accurately predict Alzheimer’s diagnosis in this dataset. The model improved conceptually when clinical assessment variables were included, showing that cognitive and functional assessment features are much stronger predictors. (in your own words)

## Feature importance

Feature importance helps us see which inputs influenced the Random Forest most.

Be careful:
- importance does **not** prove causation
- importance can be split across multiple one-hot encoded columns
- importance tells us what mattered to the model, not necessarily what matters in the real world


## Interpretation prompts

Write short answers below:
- How well did the model perform?
- Which class was easier or harder to predict?
- Which variables seemed most important?
- Where did the model make mistakes?
- How could this model be used by a real organization?
- What would you improve next?


### Student interpretation summary
Replace this section with your final written interpretation.
